# Test: Convert a Minutes Paper to Custom XML Schema

General schema format:
```
<committeeDoc>
<documentType> AP <\documentType>
<committeeName> SEN <\committeeName>
<committeeMeetingData> 2024-05-22 </committeeMeetingData>
<committeeYear> 2023/4 </committeeYear>
<meetingNumber> 3 </meetingNumber>
<items>
<agendaItem number="1" paper = "S 23/24 3A">
<metadata> ... </metadata>
<content> ... </content>
<\agendaItem>
     ...
     ...
<\items>
<\committeeDoc>
```

## Imports

In [1]:
import pymupdf
import lxml
import re
import pandas as pd

## Understand how blocks are automatically detected

In [2]:
doc = pymupdf.open("SEN_M_20240618.pdf")

page = doc[0]
blocks = page.get_text("dict", sort=True)["blocks"]
for block in blocks:
    if block["type"] == 0:
        # Get the raw text with newlines visible
        spans = [span for line in block["lines"] for span in line["spans"]]
        raw_text = " ".join(s["text"] for s in spans)
        
        print(f"=== BLOCK ===")
        print(f"Raw text repr: {repr(raw_text)}")
        print(f"Num lines: {len(block['lines'])}")
        print(f"Font size: {spans[0]['size'] if spans else 'N/A'}")
        print(f"Bold: {bool(spans[0]['flags'] & 16) if spans else 'N/A'}")
        print()

=== BLOCK ===
Raw text repr: '  Senatus Academicus  Reconvened Meeting '
Num lines: 3
Font size: 10.979999542236328
Bold: True

=== BLOCK ===
Raw text repr: '  Tuesday 18 June 2024 at 9:45-10:45am '
Num lines: 2
Font size: 10.979999542236328
Bold: False

=== BLOCK ===
Raw text repr: 'Microsoft Teams   '
Num lines: 2
Font size: 10.979999542236328
Bold: False

=== BLOCK ===
Raw text repr: 'Confirmed Minute    Attendees:  Peter Adkins, Gill Aitken, Mteeve Amugune, Arianna Andreangeli, Jonathan  Ansell, Kate Ash-Irisarri, Michael Barany, Laura Bickerton, Richard Blythe, Catherine Bovill,  Holly Branigan, Aidan Brown, Rory Callison, Jeremy Carrette, Leigh Chalmers, Neil Chue  Hong, Juan Cruz, Sarah Cunningham-Burley, Sumari Dancer, Luigi Del Debbio, Chris Dent,  Charlotte Desvages, Simone Dimartino, Claire Duncanson, Murray Earle, Tonks Fawcett,  Samantha Fawkner, Manuel Fernandez-Gotz, Chris French, Vashti Galpin, Soledad Garcia  Ferrari, Benjamin Goddard, Richard Gratwick, Colm Harmon, Ga

## Test table extraction

In [3]:
doc_with_table = pymupdf.open("SEN_AP_20240522.pdf")
page_with_table = doc_with_table[113]
table_blocks = page_with_table.get_text("dict", sort=True)["blocks"]
tables = page_with_table.find_tables()
if tables:
    # 3. Extract the first table
    my_table = tables[0]
    
    # 4. Convert to a readable Pandas DataFrame
    df = my_table.to_pandas()
    print(df)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
                                                Col0 Area of focus  Col2  \
0  Knowledge and\nengagement –\nunderstanding the...          None  None   
1  Research\ngrants/projects –\ncontinuous\nimpro...          None  None   

                                                Col3 Where we are now  Col5  \
0  Key processes and\ndocumentation are\ndifferen...             None  None   
1  Projects not set up\nquickly enough and are\nm...             None  None   

                                                Col6 Where we want to be  \
0  Business processes and\noperating procedures t...                None   
1  We have processes in place\nwhich are simple, ...                None   

   Col8                                               Col9 What we’re doing  \
0  None  • Publish standard operating procedures in\na ...             None   
1  None  • Assess and tackle backlogs to support\nset-u... 

## Create custom blocks

In [4]:
# Returns true if a block overlaps with any of the tables on a page
def overlaps_table(block_bbox, table_bboxes):
    bx0, by0, bx1, by1 = block_bbox
    for tx0, ty0, tx1, ty1 in table_bboxes:
        # If the block is entirely outside the table in any direction, no overlap
        if bx1 < tx0 or bx0 > tx1 or by1 < ty0 or by0 > ty1:
            continue
        return True
    return False

# Joins all accumulated lines into one block dict
def make_sub_block(current_lines):
    text = " ".join(l["text"] for l in current_lines) 
    text = re.sub(r'\s+', ' ', text).strip() # clean up by getting rid of extra whitespace
    if not text:
        return None  # caller should check for None and skip appending
    return {
        "text": text,
        "font_size": current_lines[0]["font_size"],
        "font": current_lines[0]["font_size"],
        "bold": current_lines[0]["bold"],
        "bbox": current_lines[0]["bbox"],
        "type": "text",
    }

# Returns whether a text block is a page number
def is_page_num(block):
    text = block["text"].strip()
    if block["type"] != "text":
        return False
    # Matches bare page numbers: "1", "12" etc.
    if re.fullmatch(r'\d+', text):
        return True
     # Matches "Page 1 of 9", "Page 12 of 100" etc. (case insensitive)
    if re.fullmatch(r'[Pp]age\s+\d+\s+of\s+\d+', text):
        return True
    return False

# Extract blocks across all pages of a document
def extract_blocks(doc):
    all_blocks = []
    for page in doc:
        blocks = page.get_text("dict", sort = True)["blocks"]
        
        # Extract tables on page and add each cell as a block
        table_bboxes = []
        for table in page.find_tables():
            table_bboxes.append(table.bbox)  # record where the table is on the page
            rows = table.extract() # returns the table as a list of rows
            
            for row_i, row in enumerate(rows):
                for col_i, cell_text in enumerate(row):
                    if not cell_text:
                        continue
                    else:
                        all_blocks.append({
                            "text": cell_text.strip(),
                            "type": "table",      
                            "row": row_i,      
                            "col": col_i,       
                            "bbox": table.bbox,   
                            "font_size": None, #font_size, font, and bold not available from find_tables
                            "font": None,
                            "bold": None,
                        })
        
        for block in blocks:
            if block["type"] == 0: # only include text (ignore images for now)
                # Skip block it overlaps w/ table on page -> content already captured
                if overlaps_table(block["bbox"], table_bboxes):
                    continue
                
                spans = [span for line in block["lines"] for span in line["spans"]]
                if not spans: 
                    continue

                # Pick the longest span as the dominant one for formatting
                dominant_span = max(spans, key = lambda s: len(s["text"]))
                dominant_size = dominant_span["size"]
                dominant_bold = bool(dominant_span["flags"] & 16)

                # Build a metadata dictionary per line
                lines_meta = []
                for line in block["lines"]:
                    line_spans = line["spans"]
                    if not line_spans:
                        continue
                    line_text = " ".join(s["text"] for s in line_spans)
                    line_dominant = max(line_spans, key=lambda s: len(s["text"])) # dominant styling per line (longest span)
                    lines_meta.append({
                        "text": line_text,
                        "bbox": line_dominant["bbox"],
                        "font": line_dominant["font"],
                        "font_size": line_dominant["size"],
                        "bold": bool(line_dominant["flags"] & 16),
                        "type": "text"
                    })

                # Walk through lines and split into sub-blocks when formatting changes
                # Sub-blocks will become independent blocks
                sub_blocks = []
                current_lines = [] #current possible lines for a sub-block
                for line in lines_meta:
                    if not current_lines:
                        current_lines.append(line)
                        continue

                    prev_line = current_lines[-1]

                    should_split = False

                    # Font size change
                    if abs(line["font_size"] - prev_line["font_size"]) > 1:
                        should_split = True

                    # Font changes
                    if line["font"] != prev_line["font"]:
                        should_split = True
        
                    # Bold state changes
                    if line["bold"] != prev_line["bold"]:
                        should_split = True

                    # Multiple newlines
                    if re.search(r'\n{2,}', line["text"]):
                        should_split = True

                    # 3+ whitespace characters
                    if re.search(r'\s{3,}', line["text"]):
                        should_split = True

                    if should_split:
                        if make_sub_block(current_lines) is not None:
                            sub_blocks.append(make_sub_block(current_lines))
                        current_lines = [line] # current line should be start of a new block
                    else:
                        current_lines.append(line) # current line should be part of ongoing block

                # Add last accumulated group
                if current_lines:
                    text = " ".join(l["text"] for l in current_lines)
                    text = re.sub(r'\s+', ' ', text).strip()
                    if text:
                        if make_sub_block(current_lines) is not None:
                            sub_blocks.append(make_sub_block(current_lines))

                for b in sub_blocks:
                    if b["text"]:
                        all_blocks.append(b)
                    
                
                
    # Merge blocks that are part of the same paragraph together
    extracted_blocks = []
    buffer = all_blocks[0] if all_blocks else None

    for block in all_blocks[1:]: # first block starts as buffer
        if is_page_num(block): #skip page nums for XML conversion
            continue
        
        curr_text = block["text"].strip()
        
        lowercase_start = curr_text[0].islower() # if next paragraph starts with lowercase, indicates continuation of the same paragraph
        list_item = re.fullmatch(r'\d+\.', buffer["text"]) # if a paragraph is a list item like 1. or 2. it should be combined with the associated text
        same_paragraph = lowercase_start or list_item
        
        if same_paragraph:
            buffer["text"] = buffer["text"] + " " + curr_text
        else:
            extracted_blocks.append(buffer)
            buffer = block

    if buffer:
        extracted_blocks.append(buffer)
    
    return extracted_blocks

In [9]:
# Print extracted blocks
for block in extract_blocks(doc_with_table[113:116]):
    print(block["text"])
    print("---")

Area of focus
---
Where we are now
---
Where we want to be
---
What we’re doing
---
Knowledge and
engagement –
understanding the
process
---
Key processes and
documentation are
different across the
organisation
---
Business processes and
operating procedures that
are consistent and easy to
understand
---
• Publish standard operating procedures in
a user-friendly format
• Have the appropriate guidelines and
training in place to help you use these
processes.
• Provide initial support to help you adopt
the processes and guidelines and
overcome any challenges
• Provide ongoing support and engagement
---
Research
grants/projects –
continuous
improvement of
finance processes
---
Projects not set up
quickly enough and are
manual, complex and
not well-understood
Difficult to manage
projects in the most
effective way.
---
We have processes in place
which are simple, well
understood and work quickly
so that we save time,
maximise the recovery of
external funds and add
more value.
---
• Assess an

## Automatically extract metadata from filename
Metadata:
* Document type
* Committee name
* Meeting date (or meeting start date and end date)
* Committee year
* Meeting number

Notes:
* Another solution is to have start date be the same as end date for single day meetings
* Meeting number cannot automatically be extracted for minutes. For agenda and papers, it can be deduced from the paper codes. But minutes do not generally include paper codes (except in reference to past/future meetings), which would be risky to extract incorrectly
* We are currently assuming that August 1st begins a new academic year

In [6]:
file_name = doc.name.removesuffix('.pdf')

# Validate file name
valid_meeting_pattern = r"^(SEN|SEC|APRC|SQAC)_(AP|M)_20\d{2}(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])$" #supports years 2000-2099
valid_e_meeting_pattern = r"^(SEN|SEC|APRC|SQAC)_(AP|M)_20\d{2}(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])_20\d{2}(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])_e$"
if not(re.match(valid_meeting_pattern, file_name) or re.match(valid_e_meeting_pattern, file_name)):
    raise Exception("Invalid file name format")

parts = file_name.split('_')
e_meeting = len(parts) > 3

# Extract committee name
extracted_committee_name = None
match parts[0]:
    case "SEN":
        extracted_committee_name = "Senate"
    case "SEC":
        extracted_committee_name = "Senate Education Committee"
    case "APRC":
        extracted_committee_name = "Academic Policy and Regulations Committee"
    case "SQAC":
        extracted_committee_name = "Senate Quality Assurance Committee"
        
# Extract document type
extracted_doc_type = None
match parts[1]:
    case "M":
        extracted_doc_type = "Minutes"
    case "AP":
        extracted_doc_type = "Agenda and Papers"
        
# Extract meeting date(s)
extracted_date = None
extracted_start_date = None
extracted_end_date = None
extracted_dates = [extracted_date, extracted_start_date, extracted_end_date]

if e_meeting:
    extracted_start_date = parts[2]
    extracted_end_date = parts[3]
else:
    extracted_date = parts[2]
    
# Extract academic year
extracted_academic_year = None
def extract_year(d):
    year = int(d[:4])
    new_academic_year = f"{year}0801" #August 1 of the year
    if d >= new_academic_year: 
        return f"{year}/{year+1}"
    return f"{year-1}/{year}"
if e_meeting:
    extracted_academic_year = extract_year(extracted_start_date)
else:
    extracted_academic_year = extract_year(extracted_date)
    
# Fix date formatting
def fix_date_formatting(d):
    if d:
        return f"{d[:4]}-{d[4:6]}-{d[6:]}"
    return None
extracted_date = fix_date_formatting(extracted_date)
extracted_start_date = fix_date_formatting(extracted_start_date)
extracted_end_date = fix_date_formatting(extracted_end_date)

# Print extracted metadata
print("Committee: " + extracted_committee_name)
print("Document type: " + extracted_doc_type)
if e_meeting:
    print("Meeting start date: " + extracted_start_date)
    print("Meeting end date: " + extracted_end_date)
else:
    print("Meeting date: " + extracted_date)
print("Academic year: " + extracted_academic_year)


Committee: Senate
Document type: Minutes
Meeting date: 2024-06-18
Academic year: 2023/2024


## Create XML document
Note:
* Including just 1 tag for date; if over multiple days, included in the following format: "2024-06-10 to 2024-06-18"

In [ ]:
from lxml import etree
root = etree.Element("committeeDoc")

# Add metadata (docType, committeeName, dates, academicYear)
metadata = etree.SubElement(root, "metadata")
doc_type = etree.SubElement(metadata, "documentType")
doc_type.text = extracted_doc_type
committee_name = etree.SubElement(metadata, "committeeName")
committee_name.text = extracted_committee_name
dates = etree.SubElement(metadata, "dates")
if e_meeting:
    dates.text = f"{extracted_start_date} to {extracted_end_date}"
else:
    dates.text = extracted_date
academic_year = etree.SubElement(metadata, "academicYear")
academic_year.text = extracted_academic_year

# Loop through extracted blocks
# Current identification --> bold = heading, anything else = bodyText
current_heading = root
for block in extract_blocks(doc):
    text = block["text"]
    font_size = block["font_size"]
    font = block["font"]
    bold = block["bold"]
    
    #Indicates heading
    if bold:
        heading = etree.SubElement(root, "heading", title = text)
        current_heading = heading
        
    else:
        # Append normal text inside the active heading element
        subtext_el = etree.SubElement(current_heading, "subtext")
        subtext_el.text = text

# View tree so far
xml_string = etree.tostring(root, pretty_print=True, encoding='utf-8').decode('utf-8')
print(xml_string)

<committeeDoc>
  <metadata>
    <documentType>Minutes</documentType>
    <committeeName>Senate</committeeName>
    <dates>2024-06-18</dates>
    <academicYear>2023/2024</academicYear>
  </metadata>
  <heading title="Senatus Academicus Reconvened Meeting">
    <subtext>Tuesday 18 June 2024 at 9:45-10:45am</subtext>
    <subtext>Microsoft Teams</subtext>
  </heading>
  <heading title="Confirmed Minute">
    <subtext>Attendees: Peter Adkins, Gill Aitken, Mteeve Amugune, Arianna Andreangeli, Jonathan Ansell, Kate Ash-Irisarri, Michael Barany, Laura Bickerton, Richard Blythe, Catherine Bovill, Holly Branigan, Aidan Brown, Rory Callison, Jeremy Carrette, Leigh Chalmers, Neil Chue Hong, Juan Cruz, Sarah Cunningham-Burley, Sumari Dancer, Luigi Del Debbio, Chris Dent, Charlotte Desvages, Simone Dimartino, Claire Duncanson, Murray Earle, Tonks Fawcett, Samantha Fawkner, Manuel Fernandez-Gotz, Chris French, Vashti Galpin, Soledad Garcia Ferrari, Benjamin Goddard, Richard Gratwick, Colm Harmon, Ga